# Decoder Probe Analysis

This notebook evaluates the frozen-HOPE2 decoder probe, exports paper-style figures, and compares Phase A (`true_only`) and Phase B (`pred_exposed`) checkpoints.

Reference style targets:

- LeWorldModel qualitative decoding panels and decoded latent rollouts: https://arxiv.org/abs/2603.19312
- Hierarchical Planning with Latent World Models decoded subgoal visualizations: https://arxiv.org/abs/2604.03208
- Local copy of the LeWM paper notes: [roadmap/lwm_paper.pdf](/gpfs/home2/scur0200/main/roadmap/references/lwm_paper.pdf)

The notebook focuses on three questions:

1. Are true HOPE2 waypoint latents visually decodable?
2. Are predicted HOPE2 waypoint latents still on a decodable manifold?
3. Can we export clean figure panels for reports or paper-style notes?

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_CWD if (NOTEBOOK_CWD / "hi_decoder_probe.py").exists() else NOTEBOOK_CWD.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pathlib import Path

import pandas as pd

from scripts.decoder_probe_notebook_utils import (
    DEFAULT_ENV_DUMP,
    load_environment_dump,
    DEFAULT_HI_CKPT,
    evaluate_bundle,
    export_diverse_waypoint_gallery,
    find_epoch_probe_checkpoints,
    load_probe_bundle,
    make_output_dir,
    sample_batch,
    run_decoder_probe,
    save_checkpoint_comparison_figure,
    save_rollout_story_figure,
    save_waypoint_panel,
    summarize_environment_dump,
    summarize_metrics,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)


## Paths

Set the Phase A and optional Phase B run directories below. The notebook will use the latest `*_probe.pt` in each run directory unless you override it explicitly.

In [ ]:
HI_CKPT = DEFAULT_HI_CKPT

PHASE_A_RUN = "/scratch-shared/scur0200/stablewm_data/runs/hi_decoder_probe_true_full_20260521_151854"
PHASE_B_RUN = None  # replace with a real Phase B run dir when available

# Optional explicit checkpoint overrides.
PHASE_A_PROBE_CKPT = None
PHASE_B_PROBE_CKPT = None
ENV_DUMP_PATH = DEFAULT_ENV_DUMP
CACHE_DIR = None  # e.g. '/scratch-shared/scur0200/stablewm_data'

EXPORT_DIR_NAME = "notebook_exports"
BATCH_INDEX = 0
MAX_EVAL_BATCHES = 8


## Load Bundles

In [ ]:
phase_a = load_probe_bundle(
    run_dir=PHASE_A_RUN,
    probe_ckpt=PHASE_A_PROBE_CKPT,
    hi_ckpt=HI_CKPT,
    cache_dir=CACHE_DIR,
)
phase_a_export_dir = make_output_dir(PHASE_A_RUN, name=EXPORT_DIR_NAME)

phase_b = None
phase_b_export_dir = None
if PHASE_B_RUN:
    phase_b = load_probe_bundle(
        run_dir=PHASE_B_RUN,
        probe_ckpt=PHASE_B_PROBE_CKPT,
        hi_ckpt=HI_CKPT,
        cache_dir=CACHE_DIR,
    )
    phase_b_export_dir = make_output_dir(PHASE_B_RUN, name=EXPORT_DIR_NAME)

print("Phase A probe:", phase_a.probe_ckpt)
if phase_b is not None:
    print("Phase B probe:", phase_b.probe_ckpt)

## Environment Dump

The training jobs currently persist the environment dump to `environment.json` in the repo root. This cell loads that dump so the notebook can display the local-node software/runtime environment used by the run.

If you later copy a different environment dump from another job, just point `ENV_DUMP_PATH` at that file.

In [ ]:
env_dump = load_environment_dump(ENV_DUMP_PATH)
display(summarize_environment_dump(env_dump).rename("environment"))
display(pd.Series(env_dump.get("slurm", {}), name="slurm"))
display(pd.Series(env_dump.get("cuda", {}), name="cuda"))

## Quantitative Checks

These metrics are the first health check. For Phase B, the most important value is whether `pixel_mse_pred` / `psnr_pred` improve while true-latent reconstruction stays stable.

In [ ]:
phase_a_df = evaluate_bundle(phase_a, split="val", max_batches=MAX_EVAL_BATCHES)
phase_a_summary = summarize_metrics(phase_a_df)
display(phase_a_df)
display(phase_a_summary.rename("phase_a_summary"))

if phase_b is not None:
    phase_b_df = evaluate_bundle(phase_b, split="val", max_batches=MAX_EVAL_BATCHES)
    phase_b_summary = summarize_metrics(phase_b_df)
    display(phase_b_df)
    display(phase_b_summary.rename("phase_b_summary"))
    compare_df = pd.DataFrame({
        "phase_a": phase_a_summary,
        "phase_b": phase_b_summary,
        "delta_b_minus_a": phase_b_summary - phase_a_summary,
    })
    display(compare_df)

## One Batch, Full Probe Outputs

This gives us the tensors used for visualization: context frames, targets, decoded true latents, decoded predicted latents, and an absolute error map.

In [ ]:
batch = sample_batch(phase_a, split="val", batch_index=BATCH_INDEX)
phase_a_outputs = run_decoder_probe(phase_a, batch)
phase_a_outputs["metrics"]

## Paper-Ready Figure 1: Waypoint Panel

This panel mirrors the most useful paper-style qualitative layout: context, target, decoded true latent, decoded predicted latent, and an error map.

In [ ]:
phase_a_panel_path = phase_a_export_dir / "phase_a_waypoint_panel.png"
save_waypoint_panel(
    phase_a_outputs,
    save_path=phase_a_panel_path,
    num_rows=4,
    title="Phase A Decoder Probe: Context / Target / Decoded True / Decoded Pred / Error",
)
phase_a_panel_path

## Paper-Ready Figure 2: Rollout Story

This is a trajectory-style figure inspired by decoded latent rollout and decoded subgoal visualizations in recent world-model papers.

In [ ]:
phase_a_story_path = phase_a_export_dir / "phase_a_rollout_story.png"
save_rollout_story_figure(
    phase_a_outputs,
    save_path=phase_a_story_path,
    sample_index=0,
    title="Phase A Waypoint Rollout Story",
)
phase_a_story_path

## Optional: Phase A vs Phase B Comparison

Once Phase B finishes, this exports a neat side-by-side comparison focused on decoded predicted latents.

In [ ]:
if phase_b is not None:
    phase_b_outputs = run_decoder_probe(phase_b, batch)
    phase_b_panel_path = phase_b_export_dir / "phase_b_waypoint_panel.png"
    save_waypoint_panel(
        phase_b_outputs,
        save_path=phase_b_panel_path,
        num_rows=4,
        title="Phase B Decoder Probe: Context / Target / Decoded True / Decoded Pred / Error",
    )

    comparison_path = phase_b_export_dir / "phase_a_vs_phase_b_pred_comparison.png"
    save_checkpoint_comparison_figure(
        [("Phase A", phase_a), ("Phase B", phase_b)],
        batch=batch,
        save_path=comparison_path,
        sample_index=0,
        title="Predicted-Latent Decoding: Phase A vs Phase B",
    )
    display({
        "phase_b_panel": str(phase_b_panel_path),
        "comparison": str(comparison_path),
    })
else:
    print("Set PHASE_B_RUN to enable Phase A vs Phase B figure export.")

## Optional: Epoch Comparison Inside One Run

This is useful when you want to show that the decoder rapidly becomes good and then mostly refines details.

In [ ]:
epoch_ckpts = find_epoch_probe_checkpoints(PHASE_A_RUN)
epoch_ckpts[:5], epoch_ckpts[-3:]

In [ ]:
selected_epoch_ckpts = []
for idx in [0, min(4, len(epoch_ckpts) - 1), len(epoch_ckpts) - 1]:
    if idx >= 0:
        selected_epoch_ckpts.append(epoch_ckpts[idx])

epoch_bundles = []
for ckpt in selected_epoch_ckpts:
    label = ckpt.stem.replace("_probe", "")
    epoch_bundles.append((label, load_probe_bundle(run_dir=PHASE_A_RUN, probe_ckpt=ckpt, hi_ckpt=HI_CKPT, cache_dir=CACHE_DIR)))

epoch_comparison_path = phase_a_export_dir / "phase_a_epoch_comparison.png"
save_checkpoint_comparison_figure(
    epoch_bundles,
    batch=batch,
    save_path=epoch_comparison_path,
    sample_index=0,
    title="Phase A Decoder Evolution Across Epochs",
)
epoch_comparison_path

## Export A 10-Example Gallery

This exports a folder with multiple different validation examples so you can browse diverse starting contexts and waypoint transitions instead of a single hand-picked panel.


In [ ]:
gallery_dir = phase_a_export_dir / "phase_a_gallery"
gallery_df = export_diverse_waypoint_gallery(
    phase_a,
    out_dir=gallery_dir,
    split="val",
    num_examples=10,
    rows_per_panel=1,
    examples_per_batch=2,
    save_story_figures=True,
)
display(gallery_df)
gallery_dir


## Export Summary

The saved figures under `notebook_exports/` are the intended paper-style outputs from this notebook.

In [ ]:
sorted(str(p) for p in phase_a_export_dir.glob("*.png"))